In [ ]:
using Random, Plots, Statistics, LinearAlgebra, BenchmarkTools, Base.Threads

In [ ]:
# Celda 3: Función con sesgo en la dirección
function random_criterion(r, bias=0.5)
    # r es un número aleatorio uniforme [0,1]
    # bias=0.5: igual probabilidad; bias=0.7: 70% chance de +1
    return r < bias ? 1.0 : -1.0
end

In [ ]:
function single_random_walk_reset(num_steps, reset_prob, dt=0.1, a=1.0, epsilon=0.1, rng=MersenneTwister())
    x = 0.0
    y = 0.0
    x_history = Vector{Float64}(undef, num_steps+1)
    y_history = Vector{Float64}(undef, num_steps+1)
    x_history[1] = x
    y_history[1] = y
    
    for i in 2:num_steps+1
        y += sqrt(2 * dt) * randn(rng)
        if y > 1
            y = 0.0
            # Modificación: usar random_criterion con sesgo
            direction = random_criterion(rand(rng), 0.3)  # Ej. bias=0.7 favorece +1
            x += direction * a
        end
        x_history[i] = x
        y_history[i] = y
    end
    return x_history, y_history
end

In [ ]:
function generate_ensemble_msd(num_walks, num_steps, reset_prob, dt=0.1, a=1.0, epsilon=0.1; seed=1234)
    nthreads = Threads.nthreads()
    # Acumuladores por hilo: sum_x2 por paso
    sum_x2_threads = [zeros(Float64, num_steps+1) for _ in 1:nthreads]
    
    @threads for i in 1:num_walks
        tid = Threads.threadid()
        rng = MersenneTwister(seed + i + 37*tid)  # RNG por hilo
        x_hist, _ = single_random_walk_reset(num_steps, reset_prob, dt, a, epsilon, rng)
        sum_x2_threads[tid] .+= x_hist .^ 2
    end
    
    # Reducir sumas de hilos
    sum_x2 = reduce(+, sum_x2_threads)
    msd = sum_x2 ./ num_walks
    return msd
end

In [ ]:
function generate_ensemble_trajectories(num_walks, num_steps, reset_prob, num_samples=100, dt=0.1, a=1.0, epsilon=0.1; seed=1234)
    # Solo guardar num_samples trayectorias
    sample_indices = collect(1:min(num_samples, num_walks))
    x_samples = Matrix{Float64}(undef, num_steps+1, length(sample_indices))
    y_samples = Matrix{Float64}(undef, num_steps+1, length(sample_indices))
    
    @threads for i in 1:num_walks
        tid = Threads.threadid()
        rng = MersenneTwister(seed + i + 37*tid)
        x_hist, y_hist = single_random_walk_reset(num_steps, reset_prob, dt, a, epsilon, rng)
        
        # Si es una muestra, guardar
        sample_pos = findfirst(==(i), sample_indices)
        if sample_pos !== nothing
            x_samples[:, sample_pos] = x_hist
            y_samples[:, sample_pos] = y_hist
        end
    end
    
    return x_samples, y_samples
end

In [ ]:
# Celda 7: Funciones de plotting
function plot_walks(x_history, y_history, num_walks=100)
    plt = plot(xlabel="X Position", ylabel="Y Position", title="Random Walks with Re-Entry at Origin", legend=false)
    for j in 1:min(num_walks, size(x_history, 2))
        plot!(plt, x_history[:, j], y_history[:, j])
    end
    return plt
end

In [ ]:
function plot_msd(msd)
    plot(1:length(msd), msd, xlabel="Time (steps)", ylabel="MSD", title="Mean Square Displacement", label=false)
end

In [ ]:
function run_and_plot_msd(steps=10000, ensemble_size=10000, num_samples=100, reset_prob=0.1, dt=0.1, a=1.0, epsilon=0.1)  # 10k steps, 10k walks, 100 samples para plotear
    msd = generate_ensemble_msd(steps, ensemble_size, reset_prob, dt, a, epsilon)
    plot_msd(msd)
end


In [ ]:
function run_and_plot_trajectories(steps=10000, ensemble_size=10000, num_samples=100, reset_prob=0.1, dt=0.1, a=1.0, epsilon=0.1)
    # Generar trayectorias para plotting (solo num_samples)
    x_samples, y_samples = generate_ensemble_trajectories(ensemble_size, steps, reset_prob, num_samples, dt, a, epsilon)
    
    # Plotear
    plot_walks(x_samples, y_samples, num_samples)
end

In [ ]:
run_and_plot_msd(10000, 10000, 1000)  # 10k steps, 10k walks, 100 samples para plotear